[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kanchukaitis/paleoCAMP/blob/main/python_for_climate_part2.ipynb)

# Introduction to Python for Paleoclimate Science — Part 2: Xarray, Matplotlib, and Cartopy

This is the second of two introductory notebooks. It picks up where Part 1 left off.

**This notebook covers:**
1. Xarray — labeled N-dimensional arrays and netCDF files
2. Matplotlib — publication-quality plotting
3. Cartopy — geographic maps and projections


## Setup: Install Cartopy and Import Packages

One benefit of using Google CoLab is that it has a consistent and compatible set of the most common packages installed.  It does not include, however, Cartopy.  Cartopy is the most recent package to interface with the plotting package Matplotlib to create maps.  Cartopy is both very much still in development and has had occassional clashes and dependency problems with CoLab.  At the moment (May 2026), the current Cartopy appears to be working ok once installed using `pip` without any additional drama.  Run the cell below in CoLab and you should be able to use it make maps. 

In [ ]:
# Cartopy is not pre-installed in Colab, so this may take 30 seconds or more
import subprocess, importlib
if importlib.util.find_spec('cartopy') is None:
    subprocess.run(['pip', 'install', 'cartopy', '-q'], check=True)
    print('Cartopy installed.')
else:
    print('Cartopy already available.')

We'll get our other packages here, too.  Nearly all my code imports Numpy, Pandas, Xarray, MatplotLib, and Cartopy. Scipy and scikit-learn are other common libraries, and there are others we'll use at paleoCAMP. 

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
import warnings

# suppress Cartopy's Natural Earth download warnings
warnings.filterwarnings('ignore')  

# check our versions
print('xarray:', xr.__version__)
print('matplotlib:', mpl.__version__)

## 1. Xarray: Labeled N-Dimensional Data

[Xarray](https://docs.xarray.dev/) is really one of the main reasons I've stuck with Python, despite my frustrations with its overall syntax.  Xarray extends NumPy arrays with **labeled dimensions and coordinates**, making it ideal for climate model output and observational grids that live in netCDF files.  What this means in practice is that the dimensions of your data (latitude, longitude, time, etc) are properly understood and tracked across methods and operations.  It really is, in my opinion, one of the best things about Python.

Xarray adds two new data structures (classes) for us to use
- A **`DataArray`** is an N-dimensional array with named dimensions and coordinate labels
- A **`Dataset`** is a collection of `DataArray`s sharing the same coordinates. So, it can hold a bunch of variables on the same grid, for instance.

DataSets mirror the structure of netCDF files, one of the most common data formats for climate model and gridded data. In DataSets, Dimensions might be `lat`, `lon`, `time`, `plev`. There could be several data variables (`T`,`U`,`V`) that share those dimensions in common.  You can use the named dimensions to operate on the data, so instead of having to remember that axis 0 is time and axis 2 is longitude, you just refer to them by their names.

At paleoCAMP we'll use actual netCDF files from gridded modern observations and climate model outputs.  Here for simplicity we'll just build an artificial DataArray (geographic coordinates for a single variable) for us to practice on.  This also has the benefit of allowing you to see the structure of DataArrays as we build it. Note that Xarray and Numpy interact well together and that I use Numpy methods to help create the Xarray DataArray object: 

In [ ]:
lons = np.arange(-178, 180, 2.0)   # create longitude points
lats = np.arange(-88, 90, 2.0)     # create latitude points
times = pd.date_range('2000-01-01', periods=3, freq='MS') # create 3 monthly time points starting with January 2000

# Make some random artificial temperature data using a random anomaly plus an imposed latitudinal gradient
lat2d, lon2d = np.meshgrid(lats, lons, indexing='ij')   # note that specifying indexing='ij' means we'll use a (lat, lon) order, which makes it easier to impose our artificial gradient
base = 15 - 0.5 * np.abs(lat2d)    # create the latitudinal structure with temperatures cooler at poles
data = base[np.newaxis, :, :] + np.random.randn(3, len(lats), len(lons)) * 2 # add some Gaussian noise for flavor

# Create a DataArray with our named dimensions and coordinates
da = xr.DataArray( # DataArray are declared with parentheses
    data, # this is our synthetic temperature field we generated above
    dims=['time', 'lat', 'lon'], # this declares the names of the dimensions that apply to the data field
    coords={'time': times, 'lat': lats, 'lon': lons}, # this assigns the arrays we generated to those named dimensions
    name='temp', # name of the data variable to which the dimension and coordinates refer
    attrs={'units': 'degrees_C', 'long_name': 'Surface Air Temperature'} # add some simple attributes
)
da # print the DataArray to screen

The pretty print of the DataArray summary above tells you some things about the object you just generated.  For instance, you can see it is a DataArray with a variable called `temp` and that variable has 3 dimensions `time` (length 3), `lat` (length 89), and `lon` (length 179).  You also get an abbreviated look at some of the temperature data itself, and then some information on the coordinates.  You can also see the DataArray's short list of attributes. 

We can ask Xarray to tell us more about individual elements of the DataArray:

In [ ]:
print('Dimensions:', da.dims) # name of the dimensions
print('Shape:', da.shape) # size of the dimensions
print('Coordinates:', list(da.coords)) # name of the coordinates
print('Attributes:', da.attrs) # the attributes

The real strength of Xarray is that it keeps track of and lets us use the coordinate values without having to know, for example, which column or row in the array they belong to (unlike regular Numpy arrays).  We can use the `.sel` methods from Xarray to select parts of the DataArray's variable (temperature) based on the values of the coordinate.  So we can grab a specific grid point of data based on which point in the DataArray is nearest to a given set of coordinates:

In [ ]:
point = da.sel(lat=31.0, lon=-110.0, method='nearest') # get the data at the point nearest to the requested coordinates
print('Time series at ~Tucson lat/lon:')
print(point.values) # using .values pulls out just the numbers without bringing the latitude, longitude, and time as well

We can also use `.sel` to get a `slice` of a multidimensional DataArray.  We still use `da.sel` and indicate a coordinate or coordinates, but we now specify a `slice` in parentheses:

In [ ]:
tropics = da.sel(lat=slice(-23.5, 23.5)) # Select a slice of latitudes (a latitude band)
print('Tropics shape:', tropics.shape)   # keep 3 time steps, but fewer lat points

jan = da.sel(time='2000-01') # Select a specific time step by value
print('January 2000 shape:', jan.shape)  # now this is just 2-D (just 1 time point and then lat x lon)

We can also calculate means and other properties over different dimensions.  Once again, unlike Numpy, we don't need to remember which `axes` correspond to which dimension.  Instead, we just specigy the name of the dimension:

In [ ]:
time_mean = da.mean(dim='time') # calculate the mean value over time
print('Time mean shape:', time_mean.shape)   # result is the mean over the 3 time points at each latitude and longitude

zonal_mean = da.mean(dim='lon') # mean over longitude
print('Zonal mean shape:', zonal_mean.shape) # we know have a zonal mean - a mean value at each time point for each latitude

We'll use DataArrays quite frequently, but we can also create DataSets, which are simply multiple DataArrays (multiple variables) in the same objeect: 

In [ ]:
precip_data = np.abs(np.random.randn(3, len(lats), len(lons))) * 5 # create some fake precipitation data
dp = xr.DataArray(precip_data, dims=['time','lat','lon'], # create a precipitation DataArray
                  coords={'time': times, 'lat': lats, 'lon': lons},
                  name='precip', attrs={'units': 'mm/day'})

ds = xr.Dataset({'temp': da, 'precip': dp}) # merge the two DataArrays into a DataSet
ds

We can access (and even extract) individual data variables in a DataSet by name, either using square brackets and quotation marks around the variable name or using the dot notation (`ds.temp`) directly. 

In [ ]:
# Accessing variables in a Dataset
print(ds['temp']) # output is a DataArray again, or we can equivalently use ds.temp


We'll often use Xarray with netCDF files, as this is the most common data format for climate model output and gridded data products.  In the code block below, we'll do a simple write of our synthetic Dataset `ds` to a netCDF, and then we'll read it back out of the netCDF using `xr.open_dataset`:

In [ ]:
# Save our synthetic dataset to netCDF
ds.to_netcdf('synthetic_climate.nc') # that's how simple it is! 
print('Saved to synthetic_climate.nc')

# Read our DataSet back from that netCDF
ds_loaded = xr.open_dataset('synthetic_climate.nc') # xr.open_dataset will be your friend
print(ds_loaded)

At paleoCAMP, you'll use `xr.open_dataset` quite a bit, and you'll learn about `xr.open_mfdataset`, which allows you to open multiple related netCDFs all at once (e.g. model outputs from two sequential time periods in two different files but using the same variables and dimensions).  

You can extract, slice, group, and operate on DataSets and DataArrays, and we'll do this at paleoCAMP.  Here's just one example - calculating a global monthly mean value for temperature, weighting properly by latitude:

In [ ]:
# Compute area-weighted global mean 
weights = np.cos(np.deg2rad(da.lat)) # weights to account for area different are the cosine of latitude
global_mean = da.weighted(weights).mean(dim=['lat','lon']) # apply the weights to our temperature DataArray, taking the mean over both latitude and longitude
print('Area-weighted global mean by month:')
print(global_mean.values.round(2)) # we're left with just the temporal dimension - monthly mean global temperatures, properly weighted

### Xarray + Pandas interoperability

Xarray is most closely related to Numpy, but Xarray and Pandas also work together. A 1-D or 2-D `DataArray` can be converted to a Pandas `Series` or `DataFrame` with `.to_series()` or `.to_dataframe()`:

In [ ]:
# Convert a 1-D DataArray (time series) to a Pandas Series
tucson_ts = da.sel(lat=31.0, lon=-110.0, method='nearest') # once again, .sel let's us specify the actual values from the coordinates
tucson_series = tucson_ts.to_series()
print(type(tucson_series))
print(tucson_series)

## 2. Matplotlib: The best of the plotting libraries

[Matplotlib](https://matplotlib.org/) is Python's primary plotting library. Its design deliberately mimics (but also impressively extends) MATLAB's plotting model, so the concepts may feel familiar to you if you've used MATLAB, but the syntax differs.

**Two approaches:**
1. **pyplot interface** (`plt.plot(...)`) — a 'function based' approach more like MATLAB
2. **Object-oriented interface** (`fig, ax = plt.subplots(); ax.plot(...)`) — object oriented like most of Python and recommended

In truth, I often mix the two methods (since I was a MATLAB programmer for so long).  We'll start with using the pyplot approach to orient you, then switch to the object-oriented approach, which is what the Matplotlib developers recommend and what you should really use.

First, let's create some simple synthetic data to plot:

In [ ]:
x = np.linspace(-np.pi, np.pi, 256)
C, S = np.cos(x), np.sin(x) # make two waves

# MATLAB-like, treats plt. like a function:
plt.figure(figsize=(7, 4))
plt.plot(x, C, label='cosine')
plt.plot(x, S, label='sine')
plt.xlabel('x')
plt.ylabel('amplitude')
plt.title('sick waves, dude')
plt.legend()
plt.show()

Note that the `label` is attached to the line data when plotted, which is then used when we call `plt.legend`.

Compare this approach to the more Pythonic object oriented way - in this approach, plot objects `fig` and `ax` are created using a call to `plt.subplots`, and then the plot design is associated with (or called ON) those objects, especially `ax`. This gives precise control over your figure, especially when using subplots.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4)) # this creates the ax object
ax.plot(x, C, label='cosine', color='red', linewidth=1.5) 
ax.plot(x, S, label='sine', color='blue', linewidth=2, linestyle='--')
ax.set_xlabel('x')
ax.set_ylabel('amplitude')
ax.set_title('Object-oriented interface')
ax.legend(loc='upper right')
plt.show()

### Subplots

Creating multiple panels are easy with `plt.subplots(nrows, ncols)`. The `ax` object then becomes a 2-D array of axes that you index into (remember they are zero-indexed):

In [ ]:
data1, data2, data3, data4 = np.random.randn(4, 100) # create some random data to plot

# declare the plot objects - axs will now have dimensions of 2 x 2 -- remember to use 0 indexing! 
fig, axs = plt.subplots(2, 2, figsize=(8, 6), layout='constrained')

# a line plot
axs[0, 0].plot(data1, linewidth=0.8, color='steelblue')
axs[0, 0].set_title('Line plot')

# a scatter plot
axs[0, 1].scatter(data1, data2, s=20, alpha=0.6, color='tomato')
axs[0, 1].set_title('Scatter plot')
axs[0, 1].set_aspect('equal')

# a histogram
axs[1, 0].hist(data3, bins=20, color='teal', edgecolor='white', alpha=0.8)
axs[1, 0].set_title('Histogram')

# fill_between, e.g. for uncertainty envelopes around paleoclimate time series
axs[1, 1].plot(x, C, color='navy')
axs[1, 1].fill_between(x, C - 0.3, C + 0.3, alpha=0.3, color='cornflowerblue',label='±0.3 envelope')
axs[1, 1].set_title('fill_between')
axs[1, 1].legend(fontsize=8)

fig.suptitle('Common plot types in Matplotlib', fontsize=13) # puts a title above all the subplots
plt.show()

For exploratory figures and for sanity checks especially, you can plot right from Xarray objects.  Xarray used Matplotlib under the hood, and passes along its understanding of dimensions to the plot (but not map projections, in this case).  I still use Matplotlib directly for publication plots, but a quick plot straight from the object is good to check that things appear the way you think they should:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Xarray: plot a lat-lon slice as a filled contour plot
jan_temp = da.sel(time='2000-01').squeeze('time') # .squeeze to make sure this becomes 2D and doesn't have a singleton dimension
jan_temp.plot.contourf(ax=axes[0], cmap='RdYlBu_r', levels=15,cbar_kwargs={'label': '°C'})
axes[0].set_title('Jan 2000 — xarray.plot.contourf')

# Pandas: plot a time series
ts = global_mean.to_series()
ts.plot(ax=axes[1], marker='o', color='steelblue')
axes[1].set_ylabel('Global mean temp (°C)')
axes[1].set_title('Time series — pandas.plot')

plt.tight_layout()
plt.show()

Matplotlib has a bunch of built-in [colormaps](https://matplotlib.org/stable/users/explain/colors/colormaps.html) and [named colors](https://matplotlib.org/stable/gallery/color/named_colors.html) you can use for plotting.  

We can also save our Matplotlib figures using `.savefig` - the format will be derived from the file suffix you specify.

In [ ]:
# Saving figures
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, S, color='navy')
ax.set_title('Example saved figure')
plt.tight_layout()

fig.savefig('my_figure.pdf')   # vector — best for publication (PDF, SVG, EPS)
fig.savefig('my_figure.png', dpi=150)   # raster — for presentations or web
print('Figures saved.')

## 3. Cartopy: Making Maps in Matplotlib

[Cartopy](https://scitools.org.uk/cartopy/docs/latest/) adds geographic map projections to Matplotlib. It replaced the older Basemap package, but it is still buggy at times and not as intuitive as one might want. 

**The core concept:** when you create a Matplotlib axes, you can give it a geographic **projection** (how the sphere is mapped to 2-D). The axes now 'carries' the geographic project information.  You then pass `transform=ccrs.PlateCarree()` to the Matplotlib plotting commands (line, scatter, contour, etc.) to tell Cartopy that your data are in normal (Cartesian) lat/lon coordinates.

One of the non-intuitive parts to me: The projection declaration controls the *map* in the axes, but the transform tells Cartopy what coordinate system the *data* are in.  Separating it like this is odd for me.  

Let's import the 3 typical components of Cartopy we'll use

In [ ]:
import cartopy # base Cartopy
import cartopy.crs as ccrs # the coordinate reference systems (projections)
import cartopy.feature as cfeature   # Natural Earth features for coastlines, political borders, etc. 

print('Cartopy version:', cartopy.__version__)

Let's make a simple global map to see some of the syntax.  We'll indicate a Robinson projection, add some coatlines, and color the land and ocean differently.  Note that we apply the projection here using the `subplot_kw={'projection': ccrs.Robinson()}` syntax in the call to `plt.subplots`:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5),subplot_kw={'projection': ccrs.Robinson()}) # ccrs is the sublibrary of Cartopy with the projection information
ax.coastlines(resolution='110m', color='black', linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor='wheat', alpha=0.5)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.4)
ax.set_global()
ax.set_title('Robinson projection')
plt.show()

We can plot a few other projections below. PlateCarree is the simplest ma[p projection (equirectangular / regular lat-lon), but confusingly shares the name of the data transformation we'll typically use. Lambert Conformal is good for mid-latitude regional maps. Mollweide / Robinson are good for global visualisations

In [ ]:
fig = plt.figure(figsize=(14, 4))
projections = [ccrs.PlateCarree(), ccrs.Mollweide(), ccrs.Orthographic(-110, 40)]
names = ['PlateCarree', 'Mollweide', 'Orthographic'] # three common projections

# apply the three projections to 3 different subplots
axes = [fig.add_subplot(1, 3, i+1, projection=proj) for i, proj in enumerate(projections)]

# now create the maps in each of the 3 subplots
for ax, name in zip(axes, names):
    ax.coastlines(resolution='110m', linewidth=0.5)
    ax.set_global()
    ax.set_title(name)

plt.tight_layout()
plt.show()

We can also create gridlines and political borders if we'd like. Note this all happens through the axes object `ax.` -- Cartopy and Matplotlib are working together to create the plot and apply the projection:

In [ ]:
fig = plt.figure(figsize=(8, 6))
projection = ccrs.LambertConformal(central_longitude=-100)
ax = fig.add_subplot(1, 1, 1, projection=projection)

# we can set the extent of the map, with the following input order: (lon_west, lon_east, lat_south, lat_north)
ax.set_extent([-130, -64, 15, 55])

ax.coastlines(color='black', linewidth=0.6)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='dimgray') # national borders from Natural Earth
ax.add_feature(cfeature.STATES, linewidth=0.3, edgecolor='lightgray') # US states borders from Natural Earth
ax.add_feature(cfeature.LAND, facecolor='wheat',alpha=0.5)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')

# the gridline specification 
gl = ax.gridlines(color='gray', linestyle=':', linewidth=0.5, draw_labels=True, x_inline=False, y_inline=False, xlocs=[-120, -100, -80], ylocs=[20, 30, 40, 50])

plt.tight_layout()
plt.show()

In paleoclimate we might want to plot the locations of our data sites, cores, etc. on a map.  We'll use a dictionary data type for this below. Note how we always  pass the `transform=` to the plotting commands. `transform=ccrs.PlateCarree()` will work for most regular latitude/longitude grids. 

In [ ]:
sites = [ # this is a Dictionary type with key-value entries
    {'name': 'Tucson',    'lon': -110.97, 'lat': 32.25},
    {'name': 'Flagstaff', 'lon': -111.65, 'lat': 35.20},
    {'name': 'Las Vegas', 'lon': -115.14, 'lat': 36.17},
    {'name': 'SNARL',     'lon': -118.83, 'lat': 37.60},
    {'name': 'Reno',      'lon': -119.77, 'lat': 39.51},
]

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(1, 1, 1,projection=ccrs.LambertConformal(central_longitude=-111))
ax.set_extent([-125, -107, 30, 40])
ax.coastlines(color='black')
ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray')
ax.add_feature(cfeature.LAND, facecolor='wheat',alpha=0.5)

for s in sites: # loop over all the entries in the sites Dictionary, using 's' as shorthand for the dictionary
    ax.plot(s['lon'], s['lat'], 'ro', markersize=6, transform=ccrs.PlateCarree())   # make sure you include transform, otherwise it won't work correctly
    ax.text(s['lon'] + 0.3, s['lat'], s['name'], fontsize=8, transform=ccrs.PlateCarree())

plt.tight_layout()
plt.show()

We'll often want to plot gridded data from observations, reanalysis, or models.  Here's an example of how to do this.  We'll use `pcolormesh` (https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.pcolormesh.html) on our artificial data from above to create a gridded visual of those data. 

In [ ]:
# Use our synthetic temperature DataArray from above
jan_temp = da.sel(time='2000-01').squeeze() # once again, .squeeze removes and singleton dimensions and flattens this to 2D

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
ax.set_global()
ax.coastlines(resolution='110m', color='black', linewidth=0.4)

im = ax.pcolormesh(
    jan_temp.lon,
    jan_temp.lat,
    jan_temp.values, # specify values here, which just gives the 'temperature' data without the dimensions, etc. 
    cmap='RdYlBu_r',
    transform=ccrs.PlateCarree()   # tell Cartopy the data are in lat/lon
)

cb = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.05, shrink=0.6, label='Temperature (°C)')
ax.set_title('Jan 2000 Surface Temperature (fake)')
plt.tight_layout()
plt.show()

Here's one more example.  Here we remove the climatological mean from our artificial temperature data and plot the anomalies using a filled contour (`.contourf`) plot.  Note that since our artificial data was a mean value function plus some noise, we're essentially just going to recover the noise as the anomalies.  

In [ ]:
clim = da.mean(dim='time').squeeze() # calculate the climatological mean over time, .squeeze to remove singleton dimension
anom = da.sel(time='2000-01').squeeze() - clim # anomaly from the long-term mean for January 2000, .squeeze to remove singleton dimension

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson(central_longitude=0))
ax.set_global()
ax.coastlines(resolution='110m', color='0.3', linewidth=0.5)
ax.add_feature(cfeature.LAND, facecolor='0.9', zorder=0)

levels = np.linspace(-4, 4, 17) # setting explicit colorbar levels gives you more control over the plot
im = ax.contourf(
    anom.lon, anom.lat, anom.values, # coordinates and values
    levels=levels, # specify the levels to use for the filled contours
    cmap='RdBu_r',  # diverging colormap that will be centered on zero because the levels are
    extend='both', # color the out-of-range values and indicate this on the colorbar with arrows
    transform=ccrs.PlateCarree()
)

gl = ax.gridlines(color='0.7', linewidth=0.4, linestyle=':') # gridlines 

cb = plt.colorbar(im, ax=ax, orientation='horizontal', pad=0.04, shrink=0.7, aspect=30) # colorbar

cb.set_label('Temperature anomaly (°C)', fontsize=10) # label the colorbar

ax.set_title('Jan 2000 Temperature Anomaly', fontsize=13)
plt.tight_layout()
plt.savefig('anomaly_map.pdf', bbox_inches='tight')
plt.show()


**Key Cartopy reminders:**
- Set the projection when creating the axes: `projection=ccrs.Robinson()`
- Pass `transform=ccrs.PlateCarree()` to plotting calls when your data are in lat/lon
- Use `ax.set_extent([lon1, lon2, lat1, lat2])` to zoom to a region

**Where to go next:**
- Xarray docs and tutorials: https://docs.xarray.dev/en/stable/tutorials-and-videos.html
- Matplotlib gallery: https://matplotlib.org/stable/gallery/index.html
- Cartopy docs: https://scitools.org.uk/cartopy/docs/latest/
- Project Pythia (Earth science Python): https://projectpythia.org/